In [1]:
import cvxpy as cp
import numpy as np
from scipy.optimize import minimize
import scipy
import os
import math
import random 
import pandas as pd
import numpy as np
import datetime as dt
from pandas_datareader import data as pdr
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import scipy.optimize as sco
from Util_def import *
from Util_model import *

import warnings
warnings.filterwarnings('ignore')


Devices:  [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU'), PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU details:  {'device_name': 'METAL'}


In [2]:
stock_list = pd.read_csv('dow30_list.csv')
stock_list = stock_list['SYMBOL'].tolist()

# 5 years data
startDate = dt.datetime(2010, 1, 1)
endDate = dt.datetime(2025, 7, 28)

start_rebalance_year = 2016  # startDate.year + 3

data = getData(stock_list, startDate, endDate)
data.fillna(method='ffill', inplace=True)
# data.fillna(method='bfill', inplace=True)
print(data.info())
avg_days = avg_days_per_month(data)

print("=" * 50)
print("Min Date:", data.index.min())
print("Max Date:", data.index.max())
print("Start Rebalance Year:", start_rebalance_year)
print(f"Average number of trading days per month: {avg_days}", "days")
print("=" * 50)


YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  30 of 30 completed

12 Failed downloads:
['IBM']: Timeout('Failed to perform, curl: (28) Operation timed out after 10001 milliseconds with 100383 bytes received. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')
['MCD']: Timeout('Failed to perform, curl: (28) Operation timed out after 10002 milliseconds with 102897 bytes received. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')
['DIS', 'V']: Timeout('Failed to perform, curl: (28) Operation timed out after 10002 milliseconds with 0 bytes received. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')
['HON', 'MMM', 'CSCO', 'HD', 'AAPL']: Timeout('Failed to perform, curl: (28) Connection timed out after 10001 milliseconds. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.')
['BA']: Timeout('Failed to perform, curl: (28) Connection timed out after 10000 milliseconds. See https://curl.se/li

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 3914 entries, 2010-01-04 to 2025-07-25
Data columns (total 30 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   AAPL    0 non-null      float64
 1   AMGN    3914 non-null   float64
 2   AMZN    3914 non-null   float64
 3   AXP     3914 non-null   float64
 4   BA      0 non-null      float64
 5   CAT     3914 non-null   float64
 6   CRM     3914 non-null   float64
 7   CSCO    0 non-null      float64
 8   CVX     3914 non-null   float64
 9   DIS     0 non-null      float64
 10  GS      3914 non-null   float64
 11  HD      0 non-null      float64
 12  HON     0 non-null      float64
 13  IBM     0 non-null      float64
 14  JNJ     0 non-null      float64
 15  JPM     3914 non-null   float64
 16  KO      3914 non-null   float64
 17  MCD     0 non-null      float64
 18  MMM     0 non-null      float64
 19  MRK     3914 non-null   float64
 20  MSFT    0 non-null      float64
 21  NKE     3914 non-nu

# QWEN 1

In [ ]:
def portfolio_optimization(
    expected_returns,
    cov_matrix,
    historical_returns=None,
    risk_free_rate=0.0,
    target_return=0.0,
    obj_func='sharpe',
    max_weight=0.3,
    min_assets=5,
    max_assets=10
):
    """
    Perform mean-variance portfolio optimization with customizable objective function and constraints.
    
    Parameters:
    - expected_returns (np.array): Expected returns for each asset.
    - cov_matrix (np.array): Covariance matrix of asset returns.
    - historical_returns (np.array, optional): Historical returns for Sortino ratio calculation (n_assets x n_periods).
    - risk_free_rate (float): Risk-free rate for Sharpe ratio calculation.
    - target_return (float): Minimum acceptable return (MAR) for Sortino ratio.
    - obj_func (str): Objective function to optimize. Options: 'return', 'sharpe', 'sortino', 'volatility'.
    - max_weight (float): Maximum weight for any single asset (e.g., 0.3 for 30%).
    - min_assets (int): Minimum number of assets in portfolio.
    - max_assets (int): Maximum number of assets in portfolio.
    
    Returns:
    - np.array: Optimal portfolio weights.
    """
    n_assets = len(expected_returns)
    
    # Validate constraints
    if min_assets > n_assets:
        raise ValueError(f"min_assets ({min_assets}) cannot exceed number of assets ({n_assets})")
    if max_assets < min_assets:
        max_assets = min_assets
    if max_assets > n_assets:
        max_assets = n_assets
    
    # Handle 'return' and 'volatility' objectives with mixed-integer programming
    if obj_func in ['return', 'volatility']:
        # Define variables
        w = cp.Variable(n_assets)
        z = cp.Variable(n_assets, boolean=True)
        
        # Constraints
        constraints = [
            w >= 0,
            w <= max_weight * z,
            cp.sum(z) >= min_assets,
            cp.sum(z) <= max_assets,
            cp.sum(w) == 1
        ]
        
        # Objective function
        if obj_func == 'return':
            objective = cp.Maximize(expected_returns @ w)
        else:  # 'volatility'
            objective = cp.Minimize(w.T @ cov_matrix @ w)
        
        # Solve the problem
        try:
            prob = cp.Problem(objective, constraints)
            prob.solve(solver=cp.CBC)  # CBC solver for mixed-integer
        except:
            try:
                prob.solve(solver=cp.GLPK_MI)  # Fallback to GLPK
            except:
                raise RuntimeError("No suitable solver available for mixed-integer optimization")
        
        if prob.status != 'optimal':
            raise RuntimeError(f"Optimization failed with status: {prob.status}")
        
        return w.value
    
    # Handle 'sharpe' and 'sortino' objectives with heuristic approach
    else:
        # Define objective function for initial optimization
        if obj_func == 'sharpe':
            def obj_func_scipy(w):
                ret = expected_returns @ w
                std = np.sqrt(w.T @ cov_matrix @ w)
                if std < 1e-8:  # Avoid division by zero
                    return float('inf')
                sharpe = (ret - risk_free_rate) / std
                return -sharpe  # Minimize negative Sharpe
        
        elif obj_func == 'sortino':
            if historical_returns is None:
                raise ValueError("historical_returns required for Sortino ratio")
            MAR = target_return
            def obj_func_scipy(w):
                port_returns = historical_returns @ w
                excess = MAR - port_returns
                downside = np.maximum(excess, 0)
                downside_variance = np.mean(downside**2)
                downside_dev = np.sqrt(downside_variance)
                if downside_dev < 1e-8:  # Avoid division by zero
                    return float('inf')
                expected_ret = np.mean(port_returns)
                sortino = (expected_ret - MAR) / downside_dev
                return -sortino  # Minimize negative Sortino
        
        # Initial optimization without cardinality constraints
        bounds = [(0, max_weight) for _ in range(n_assets)]
        cons = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
        w0 = np.ones(n_assets) / n_assets
        
        try:
            res = minimize(obj_func_scipy, w0, method='SLSQP', bounds=bounds, constraints=cons)
        except Exception as e:
            raise RuntimeError(f"Initial optimization failed: {str(e)}")
        
        if not res.success:
            raise RuntimeError(f"Initial optimization failed: {res.message}")
        
        # Select top assets based on weights
        epsilon = 1e-6
        non_zero = np.where(res.x > epsilon)[0]
        sorted_indices = np.argsort(res.x)[::-1]
        
        # Ensure at least min_assets assets are selected
        selected_assets = sorted_indices[:max_assets]
        if len(selected_assets) < min_assets:
            selected_assets = sorted_indices[:min_assets]
        
        # Extract subset for re-optimization
        selected_returns = expected_returns[selected_assets]
        selected_cov = cov_matrix[np.ix_(selected_assets, selected_assets)]
        if obj_func == 'sortino':
            selected_historical = historical_returns[:, selected_assets]
        
        # Define objective for selected assets
        if obj_func == 'sharpe':
            def obj_func_selected(w_selected):
                ret = selected_returns @ w_selected
                std = np.sqrt(w_selected.T @ selected_cov @ w_selected)
                if std < 1e-8:
                    return float('inf')
                sharpe = (ret - risk_free_rate) / std
                return -sharpe
        
        elif obj_func == 'sortino':
            def obj_func_selected(w_selected):
                port_returns = selected_historical @ w_selected
                excess = MAR - port_returns
                downside = np.maximum(excess, 0)
                downside_variance = np.mean(downside**2)
                downside_dev = np.sqrt(downside_variance)
                if downside_dev < 1e-8:
                    return float('inf')
                expected_ret = np.mean(port_returns)
                sortino = (expected_ret - MAR) / downside_dev
                return -sortino
        
        # Re-optimize for selected assets
        bounds_selected = [(0, max_weight) for _ in range(len(selected_assets))]
        cons_selected = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
        w0_selected = np.ones(len(selected_assets)) / len(selected_assets)
        
        try:
            res_selected = minimize(obj_func_selected, w0_selected, method='SLSQP', 
                                   bounds=bounds_selected, constraints=cons_selected)
        except Exception as e:
            raise RuntimeError(f"Re-optimization failed: {str(e)}")
        
        if not res_selected.success:
            raise RuntimeError(f"Re-optimization failed: {res_selected.message}")
        
        # Construct final weights vector
        final_weights = np.zeros(n_assets)
        final_weights[selected_assets] = res_selected.x
        return final_weights

# คำนวณผลตอบแทนรายวันและปรับเป็นรายปี
returns = data.pct_change().dropna()
expected_returns = returns.mean() * 252  # Annualized expected returns
cov_matrix = returns.cov() * 252        # Annualized covariance matrix

# ตั้งค่าพารามิเตอร์
risk_free_rate = 0.02  # อัตราดอกเบี้ยปลอดความเสี่ยง (2% ต่อปี)
historical_returns = returns.values  # ใช้สำหรับ Sortino ratio

# ตัวอย่างการเพิ่มตัวเลือก obj_func
obj_func = 'sharpe'  # 'sharpe', 'sortino', 'return', 'volatility'

# คำนวณพอร์ตโฟลิโอ
weights = portfolio_optimization(
    expected_returns=expected_returns.values,
    cov_matrix=cov_matrix.values,
    historical_returns=historical_returns,
    risk_free_rate=risk_free_rate,
    obj_func=obj_func,
    max_weight=0.3,
    min_assets=5,
    max_assets=10
)

# แสดงผลลัพธ์
selected_indices = np.where(weights > 1e-6)[0]
selected_stocks = [stock_list[i] for i in selected_indices]
print("\n" + "="*50)
print(f"Optimization Result ({obj_func.upper()}):")
print(f"Number of assets: {len(selected_stocks)}")
print(f"Selected stocks: {selected_stocks}")
print("\nPortfolio Weights:")
for i, stock in enumerate(selected_stocks):
    print(f"{stock}: {weights[selected_indices[i]]:.4f}")
print("="*50)

# ตรวจสอบข้อจำกัด
print("\nConstraint Check:")
print(f"Total weight: {np.sum(weights):.4f} (should be 1.0)")
print(f"Max weight: {np.max(weights):.4f} (should be <= 0.3)")
print(f"Number of assets: {np.sum(weights > 1e-6)} (should be 5-10)")
print("="*50)

# QWEN 2

In [ ]:
# 2. คำนวณผลตอบแทนและสถิติ
returns = data.pct_change().dropna()
mu = returns.mean().values
Sigma = returns.cov().values
R = returns.values  # ใช้สำหรับ Sortino
n = len(mu)

# 3. ฟังก์ชัน Optimization (แก้ไขแล้ว)
def optimize_portfolio(objective='MaxReturn', risk_free_rate=0.0, target_return=0.0):
    """
    คำนวณพอร์ตโฟลิโอที่ดีที่สุดด้วยเงื่อนไขที่กำหนด
    
    Parameters:
    objective: 'MaxReturn', 'MinVolatility', 'Sharpe', 'Sortino'
    risk_free_rate: อัตราผลตอบแทนปราศจากความเสี่ยง (สำหรับ Sharpe)
    target_return: ผลตอบแทนเป้าหมาย (สำหรับ Sortino)
    
    Returns:
    weights: น้ำหนักพอร์ตโฟลิโอ
    """
    best_weights = None
    best_value = -np.inf if objective in ['MaxReturn', 'Sharpe', 'Sortino'] else np.inf
    
    # กรณี 1: Max Return หรือ Min Volatility (ใช้ Mixed-Integer Programming)
    if objective in ['MaxReturn', 'MinVolatility']:
        # ตรวจสอบว่าใช้ CVXPY เวอร์ชันที่รองรับ boolean variables
        for k in range(5, 11):  # วนลูปจำนวนหุ้น 5-10 ตัว
            w = cp.Variable(n)
            # แก้ไขตรงนี้: ใช้ boolean=True แทน cp.Boolean
            z = cp.Variable(n, boolean=True)
            
            constraints = [
                cp.sum(w) == 1,
                w >= 0,
                w <= 0.3 * z,  # น้ำหนัก ≤ 30% ต่อหุ้น
                cp.sum(z) == k   # เลือกหุ้น k ตัว
            ]
            
            if objective == 'MaxReturn':
                prob = cp.Problem(cp.Maximize(mu.T @ w), constraints)
            else:  # MinVolatility
                prob = cp.Problem(cp.Minimize(cp.quad_form(w, Sigma)), constraints)
            
            try:
                # ใช้ ECOS_BB สำหรับ Mixed-Integer
                prob.solve(solver=cp.ECOS_BB, verbose=False, max_iters=500)
            except cp.error.SolverError:
                try:
                    # ใช้ CBC เป็น fallback solver (ต้องติดตั้งด้วย pip install cylp)
                    prob.solve(solver=cp.CBC, verbose=False)
                except:
                    continue
            except Exception as e:
                continue
                
            if prob.status not in ["infeasible", "unbounded"] and w.value is not None:
                weights = w.value
                # ตรวจสอบเงื่อนไขเพิ่มเติม
                if np.isclose(np.sum(weights), 1) and all(w <= 0.3 for w in weights):
                    # คำนวณค่า Objective
                    if objective == 'MaxReturn':
                        value = mu.T @ weights
                    else:
                        value = np.sqrt(weights.T @ Sigma @ weights)
                    
                    # อัปเดตผลลัพธ์ที่ดีที่สุด
                    if (objective == 'MaxReturn' and value > best_value) or \
                       (objective == 'MinVolatility' and value < best_value):
                        best_value = value
                        best_weights = weights
    
    # กรณี 2: Sharpe Ratio หรือ Sortino Ratio (ใช้ Heuristic Approach)
    elif objective in ['Sharpe', 'Sortino']:
        # ขั้นตอน 1: หา Optimal Weights แบบ Continuous (ไม่จำกัดจำนวนหุ้น)
        def objective_func(weights):
            port_return = weights @ mu
            if objective == 'Sharpe':
                port_vol = np.sqrt(weights @ Sigma @ weights)
                return - (port_return - risk_free_rate) / port_vol  # ลบเพราะ minimize
            else:  # Sortino
                port_returns = R @ weights
                downside = np.minimum(port_returns - target_return, 0)
                downside_risk = np.sqrt(np.mean(downside**2))
                return - (port_return - target_return) / (downside_risk + 1e-10)
        
        # Constraints
        constraints = (
            {'type': 'eq', 'fun': lambda x: np.sum(x) - 1},
            {'type': 'ineq', 'fun': lambda x: 0.3 - x}  # น้ำหนัก ≤ 30%
        )
        bounds = [(0, 0.3) for _ in range(n)]
        
        # หาจุดเริ่มต้น (Equal Weight สำหรับหุ้นทั้งหมด)
        init_weights = np.ones(n) / n
        
        result = minimize(
            objective_func,
            init_weights,
            method='SLSQP',
            bounds=bounds,
            constraints=constraints
        )
        
        if result.success:
            weights = result.x
            # ขั้นตอน 2: เลือกหุ้น 5-10 ตัวที่มีน้ำหนักสูงสุด
            top_indices = np.argsort(-weights)[:10]  # เลือกสูงสุด 10 ตัว
            if len(top_indices) < 5:  # ถ้าน้อยกว่า 5 ตัว ให้เพิ่ม
                top_indices = np.argsort(-weights)[:5]
            
            # ขั้นตอน 3: Re-optimize ด้วยหุ้นที่เลือก
            m = len(top_indices)
            mu_sub = mu[top_indices]
            Sigma_sub = Sigma[np.ix_(top_indices, top_indices)]
            R_sub = R[:, top_indices]
            
            def sub_objective_func(sub_weights):
                port_return = sub_weights @ mu_sub
                if objective == 'Sharpe':
                    port_vol = np.sqrt(sub_weights @ Sigma_sub @ sub_weights)
                    return - (port_return - risk_free_rate) / port_vol
                else:
                    port_returns = R_sub @ sub_weights
                    downside = np.minimum(port_returns - target_return, 0)
                    downside_risk = np.sqrt(np.mean(downside**2))
                    return - (port_return - target_return) / (downside_risk + 1e-10)
            
            sub_constraints = (
                {'type': 'eq', 'fun': lambda x: np.sum(x) - 1},
                {'type': 'ineq', 'fun': lambda x: 0.3 - x}
            )
            sub_bounds = [(0, 0.3) for _ in range(m)]
            
            sub_result = minimize(
                sub_objective_func,
                np.ones(m)/m,
                method='SLSQP',
                bounds=sub_bounds,
                constraints=sub_constraints
            )
            
            if sub_result.success:
                final_weights = np.zeros(n)
                final_weights[top_indices] = sub_result.x
                best_weights = final_weights
    
    # ตรวจสอบผลลัพธ์
    if best_weights is None:
        # Fallback ถ้า Mixed-Integer ล้มเหลว
        if objective in ['MaxReturn', 'MinVolatility']:
            print(f"Warning: Falling back to continuous optimization for {objective}")
            w = cp.Variable(n)
            constraints = [
                cp.sum(w) == 1,
                w >= 0,
                w <= 0.3
            ]
            
            if objective == 'MaxReturn':
                prob = cp.Problem(cp.Maximize(mu.T @ w), constraints)
            else:
                prob = cp.Problem(cp.Minimize(cp.quad_form(w, Sigma)), constraints)
            
            try:
                prob.solve()
                if w.value is not None:
                    best_weights = w.value
            except:
                pass
        
        if best_weights is None:
            raise ValueError(f"Optimization failed for {objective} objective")
    
    return pd.Series(best_weights, index=data.columns)


# เลือก Objective ที่ต้องการ
objectives = ['MaxReturn', 'MinVolatility', 'Sharpe', 'Sortino']

for obj in objectives:
    try:
        weights = optimize_portfolio(objective=obj)
        
        # แสดงผลลัพธ์
        print(f"\n{obj} Portfolio Optimization Results:")
        num_stocks = np.sum(weights > 1e-5)
        print(f"Total Stocks: {num_stocks} (should be between 5-10)")
        print(f"Max Weight: {weights.max():.2%}")
        print(f"Sum of Weights: {weights.sum():.4f}")
        
        # ตรวจสอบเงื่อนไขจำนวนหุ้น
        if not (5 <= num_stocks <= 10):
            print(f"WARNING: Number of stocks ({num_stocks}) violates constraint!")
        
        # แสดงเฉพาะหุ้นที่มีน้ำหนัก > 0
        selected = weights[weights > 1e-5].sort_values(ascending=False)
        print("\nSelected Stocks (Top 10):")
        print(selected.head(10).apply(lambda x: f"{x:.2%}"))
        
    except Exception as e:
        print(f"Error in {obj}: {str(e)}")

# GPT

In [ ]:
def _prepare_moments(prices: pd.DataFrame, rf_annual: float = 0.02, mar_annual: float = 0.0):
    """
    Compute annualized mean vector (mu), covariance (Sigma), and downside covariance (Sigma_down).
    prices: wide DF of daily close prices (already aligned & ffilled).
    """
    # daily simple returns (drop rows with any NaN to align sample)
    rets = prices.pct_change().dropna(how='any')
    ann = 252.0

    mu = rets.mean().values * ann
    Sigma = rets.cov().values * ann

    # downside covariance for Sortino (relative to Minimum Acceptable Return, MAR)
    mar_daily = mar_annual / ann
    downside = (rets - mar_daily).clip(upper=0.0)  # keep only negative deviations vs MAR
    # sample covariance of the clipped series (annualized)
    Sigma_down = np.cov(downside.values.T, bias=False) * ann

    # risk-free (annual)
    rf = rf_annual
    return rets, mu, Sigma, Sigma_down, rf


def _stats(w, mu, Sigma, rf=0.0, Sigma_down=None, mar=0.0):
    """Return dict of expected return, volatility, Sharpe, Sortino (annualized)."""
    if w is None:
        return None
    w = np.asarray(w).reshape(-1)
    exp_ret = float(mu @ w)
    var = float(w.T @ Sigma @ w)
    vol = np.sqrt(max(var, 0.0))

    # Sharpe: (E[R]-rf)/sigma
    sharpe = (exp_ret - rf) / vol if vol > 0 else -np.inf

    # Sortino: (E[R]-MAR)/downside_dev
    sortino = None
    if Sigma_down is not None:
        dvar = float(w.T @ Sigma_down @ w)
        ddev = np.sqrt(max(dvar, 0.0))
        sortino = (exp_ret - mar) / ddev if ddev > 0 else -np.inf

    return dict(exp_return=exp_ret, volatility=vol, sharpe=sharpe, sortino=sortino)


def _build_common_constraints(w, z, max_weight, min_assets, max_assets, eps=1e-4):
    """
    Common constraints:
      - budget: sum w = 1
      - long only: w >= 0
      - linking: w <= max_weight * z and w >= eps * z (avoid ghost selections)
      - cardinality: min_assets <= sum z <= max_assets
    """
    cons = [
        cp.sum(w) == 1,
        w >= eps * z,
        w >= 0,
        w <= max_weight * z,
        cp.sum(z) >= min_assets,
        cp.sum(z) <= max_assets,
    ]
    return cons


def _solve_max_return(mu, n, max_weight, min_assets, max_assets, solver='ECOS_BB', eps=1e-4):
    w = cp.Variable(n)
    z = cp.Variable(n, boolean=True)
    cons = _build_common_constraints(w, z, max_weight, min_assets, max_assets, eps)
    prob = cp.Problem(cp.Maximize(mu @ w), cons)
    prob.solve(solver=solver, mi_max_iters=100000, verbose=False)
    return w.value, z.value, prob


def _solve_min_risk(mu, Sigma, n, max_weight, min_assets, max_assets,
                    target_return=None, solver='ECOS_BB', eps=1e-4):
    """
    Minimize variance subject to optional target_return (for Sharpe grid search).
    """
    w = cp.Variable(n)
    z = cp.Variable(n, boolean=True)
    cons = _build_common_constraints(w, z, max_weight, min_assets, max_assets, eps)
    if target_return is not None:
        cons += [mu @ w >= target_return]
    risk = cp.quad_form(w, Sigma)
    prob = cp.Problem(cp.Minimize(risk), cons)
    prob.solve(solver=solver, mi_max_iters=100000, verbose=False)
    return w.value, z.value, prob


def _solve_min_downside(mu, Sigma_down, n, max_weight, min_assets, max_assets,
                        target_return=None, solver='ECOS_BB', eps=1e-4):
    """
    Minimize downside variance (for Sortino grid search) with optional target_return.
    """
    w = cp.Variable(n)
    z = cp.Variable(n, boolean=True)
    cons = _build_common_constraints(w, z, max_weight, min_assets, max_assets, eps)
    if target_return is not None:
        cons += [mu @ w >= target_return]
    risk = cp.quad_form(w, Sigma_down)
    prob = cp.Problem(cp.Minimize(risk), cons)
    prob.solve(solver=solver, mi_max_iters=100000, verbose=False)
    return w.value, z.value, prob


def _extreme_returns(mu, n, max_weight, min_assets, max_assets, solver='ECOS_BB', eps=1e-4):
    """Find min and max achievable returns under constraints (used to build the target grid)."""
    # max return
    w_max, z_max, p1 = _solve_max_return(mu, n, max_weight, min_assets, max_assets, solver=solver, eps=eps)
    if w_max is None:
        return None, None

    # min return (same as max of negative mu)
    w = cp.Variable(n)
    z = cp.Variable(n, boolean=True)
    cons = _build_common_constraints(w, z, max_weight, min_assets, max_assets, eps)
    prob = cp.Problem(cp.Minimize(mu @ w), cons)
    prob.solve(solver=solver, mi_max_iters=100000, verbose=False)
    w_min = w.value
    if w_min is None:
        return None, None

    r_max = float(mu @ w_max)
    r_min = float(mu @ w_min)
    return r_min, r_max


# ---------- Public API ----------
def optimize_portfolio(
    prices: pd.DataFrame,
    objective: str = 'sharpe',    # 'max_return' | 'min_vol' | 'sharpe' | 'sortino'
    rf_annual: float = 0.02,
    mar_annual: float = 0.00,     # for Sortino denominator; often 0 or rf
    min_assets: int = 5,
    max_assets: int = 10,
    max_weight: float = 0.30,
    n_grid: int = 25,             # grid size for Sharpe/Sortino (target-return sweep)
    solver: str = 'ECOS_BB',
    eps: float = 1e-4,
    preselect_n: int | None = None # optional: speed up via preselect top-N by indiv. Sharpe
):
    """
    Return dict with weights (pd.Series), selected tickers, and portfolio stats.
    """
    assert 0 < max_weight <= 1, "max_weight should be in (0,1]"
    assert 1 <= min_assets <= max_assets, "min_assets <= max_assets required"
    assert set(prices.columns) == set(prices.columns.unique()), "Duplicate column names detected."

    # optional preselection by individual Sharpe to speed up MIQP
    work = prices.copy()
    if preselect_n is not None and preselect_n < work.shape[1]:
        daily = work.pct_change().dropna(how='any')
        rf_daily = rf_annual / 252.0
        indiv_mu = daily.mean()
        indiv_sd = daily.std().replace(0, np.nan)
        indiv_sharpe = (indiv_mu - rf_daily) / indiv_sd
        top_cols = indiv_sharpe.sort_values(ascending=False).head(preselect_n).index.tolist()
        work = work[top_cols]

    tickers = list(work.columns)
    _, mu, Sigma, Sigma_down, rf = _prepare_moments(work, rf_annual=rf_annual, mar_annual=mar_annual)
    n = len(tickers)

    if objective.lower() in ['max_return', 'return', 'maximize return']:
        w_opt, z_opt, prob = _solve_max_return(mu, n, max_weight, min_assets, max_assets, solver=solver, eps=eps)

    elif objective.lower() in ['min_vol', 'min volatility', 'minimise volatility', 'minimize volatility']:
        w_opt, z_opt, prob = _solve_min_risk(mu, Sigma, n, max_weight, min_assets, max_assets,
                                             target_return=None, solver=solver, eps=eps)

    elif objective.lower() in ['sharpe', 'max_sharpe', 'maximize sharpe']:
        # grid search on target return, choose highest out-of-sample Sharpe
        r_min, r_max = _extreme_returns(mu, n, max_weight, min_assets, max_assets, solver=solver, eps=eps)
        if r_min is None or r_max is None:
            return {"status": "infeasible", "message": "Cannot find feasible return bounds."}
        targets = np.linspace(r_min, r_max, n_grid)

        best = {"score": -np.inf, "w": None, "z": None}
        for r_t in targets:
            w_try, z_try, _ = _solve_min_risk(mu, Sigma, n, max_weight, min_assets, max_assets,
                                              target_return=r_t, solver=solver, eps=eps)
            if w_try is None:
                continue
            s = _stats(w_try, mu, Sigma, rf=rf, Sigma_down=Sigma_down, mar=mar_annual)
            if s is None:
                continue
            if s["sharpe"] > best["score"]:
                best = {"score": s["sharpe"], "w": w_try, "z": z_try}
        w_opt, z_opt, prob = best["w"], best["z"], None

    elif objective.lower() in ['sortino', 'max_sortino', 'maximize sortino']:
        # grid search on target return, minimize downside risk and pick best Sortino
        r_min, r_max = _extreme_returns(mu, n, max_weight, min_assets, max_assets, solver=solver, eps=eps)
        if r_min is None or r_max is None:
            return {"status": "infeasible", "message": "Cannot find feasible return bounds."}
        targets = np.linspace(r_min, r_max, n_grid)

        best = {"score": -np.inf, "w": None, "z": None}
        for r_t in targets:
            w_try, z_try, _ = _solve_min_downside(mu, Sigma_down, n, max_weight, min_assets, max_assets,
                                                  target_return=r_t, solver=solver, eps=eps)
            if w_try is None:
                continue
            s = _stats(w_try, mu, Sigma, rf=rf, Sigma_down=Sigma_down, mar=mar_annual)
            if s is None or s["sortino"] is None:
                continue
            if s["sortino"] > best["score"]:
                best = {"score": s["sortino"], "w": w_try, "z": z_try}
        w_opt, z_opt, prob = best["w"], best["z"], None

    else:
        raise ValueError("objective must be one of: 'max_return', 'sharpe', 'sortino', 'min_vol'")

    if w_opt is None:
        return {"status": "infeasible", "message": "No feasible solution returned by solver."}

    # package outputs
    w_series = pd.Series(w_opt, index=tickers)
    # keep only positive-weight names for readability
    w_nonzero = w_series[w_series > 1e-6].sort_values(ascending=False)

    stats = _stats(w_series.values, mu, Sigma, rf=rf, Sigma_down=Sigma_down, mar=mar_annual)
    out = {
        "status": "optimal",
        "objective": objective,
        "weights_all": w_series,
        "weights": w_nonzero,
        "tickers": w_nonzero.index.tolist(),
        "n_assets": (w_nonzero > 0).sum(),
        "constraints": {
            "long_only": True,
            "max_weight": max_weight,
            "cardinality_range": [min_assets, max_assets],
        },
        "stats_annual": stats,  # exp_return, volatility, sharpe, sortino
        "solver": "ECOS_BB",
    }
    return out

In [ ]:
# สมมติคุณมี DataFrame 'data' จากโค้ดในคำถาม (daily close prices, ffilled แล้ว)
# เลือก Objective ได้ตามต้องการ: 'max_return' | 'sharpe' | 'sortino' | 'min_vol'
# หมายเหตุ: เป็น Mixed-Integer Optimization; ถ้าใช้ชุดหุ้นใหญ่ อาจใช้เวลาเพิ่มขึ้น

# --- ตัวอย่าง Sharpe ---
res = optimize_portfolio(
    prices=data,
    objective='sharpe',
    rf_annual=0.02,        # risk-free ต่อปี
    mar_annual=0.00,       # สำหรับ Sortino (ไม่ใช้ใน Sharpe)
    min_assets=5,
    max_assets=10,
    max_weight=0.30,
    n_grid=25,
    solver='ECOS_BB',
    eps=1e-4,
    preselect_n=None       # ตั้งเป็นเลข (เช่น 15) ถ้าต้องการเร่งความเร็วด้วยการคัดหุ้นเบื้องต้น
)
print("Maximize Sharpe Ratio")
print("EQ:", len(res["tickers"]))
print("Total Weights:", res["weights"].sum())
print("Selected tickers:", res["tickers"])
print("Weights:\n", res["weights"])
print("Annual stats:", res["stats_annual"])

# --- ตัวอย่าง Min Volatility ---
res = optimize_portfolio(
    prices=data,
    objective='min_vol',
    min_assets=5, max_assets=10, max_weight=0.30
)
print("Min Volatility")
print("EQ:", len(res["tickers"]))
print("Total Weights:", res["weights"].sum())
print("Selected tickers:", res["tickers"])
print("Weights:\n", res["weights"])
print("Annual stats:", res["stats_annual"])

# --- ตัวอย่าง Max Return ---
res = optimize_portfolio(
    prices=data,
    objective='max_return',
    min_assets=5, max_assets=10, max_weight=0.30
)
print("Max Return")
print("EQ:", len(res["tickers"]))
print("Total Weights:", res["weights"].sum())
print("Selected tickers:", res["tickers"])
print("Weights:\n", res["weights"])
print("Annual stats:", res["stats_annual"])

# --- ตัวอย่าง Max Sortino ---
res = optimize_portfolio(
    prices=data,
    objective='sortino',
    rf_annual=0.02,
    mar_annual=0.00,  # หรือใช้ rf_annual ก็ได้
    min_assets=5, max_assets=10, max_weight=0.30
)
print("Max Sortino")
print("EQ:", len(res["tickers"]))
print("Total Weights:", res["weights"].sum())
print("Selected tickers:", res["tickers"])
print("Weights:\n", res["weights"])
print("Annual stats:", res["stats_annual"])

In [ ]:
# --- ตัวอย่าง Max Return ---
res = optimize_portfolio(
    prices=data,
    objective='max_return',
    min_assets=5, max_assets=10, max_weight=0.30
)
print("Selected tickers:", res["tickers"])
print("Weights:\n", res["weights"])
print("Annual stats:", res["stats_annual"])
print(res["stats_annual"])

# DS

In [ ]:
# Calculate returns and statistics
returns = data.pct_change().dropna()
mean_returns = returns.mean()
cov_matrix = returns.cov()
risk_free_rate = 0.02  # Assume 2% risk-free rate

# Portfolio optimization functions
def portfolio_annualized_performance(weights, mean_returns, cov_matrix):
    returns = np.sum(mean_returns * weights) * 252
    std = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights))) * np.sqrt(252)
    return returns, std

def portfolio_sharpe_ratio(weights, mean_returns, cov_matrix, risk_free_rate):
    p_ret, p_std = portfolio_annualized_performance(weights, mean_returns, cov_matrix)
    return (p_ret - risk_free_rate) / p_std

def portfolio_sortino_ratio(weights, returns, risk_free_rate):
    # Calculate portfolio returns
    portfolio_returns = np.dot(returns, weights)
    
    # Calculate downside deviation
    downside_returns = portfolio_returns.copy()
    downside_returns[downside_returns > 0] = 0
    downside_deviation = np.std(downside_returns) * np.sqrt(252)
    
    # Calculate annualized return
    annualized_return = np.mean(portfolio_returns) * 252
    
    # Sortino ratio
    sortino_ratio = (annualized_return - risk_free_rate) / downside_deviation
    return sortino_ratio

def negative_sharpe_ratio(weights, mean_returns, cov_matrix, risk_free_rate):
    return -portfolio_sharpe_ratio(weights, mean_returns, cov_matrix, risk_free_rate)

def negative_sortino_ratio(weights, returns, risk_free_rate):
    return -portfolio_sortino_ratio(weights, returns, risk_free_rate)

def portfolio_volatility(weights, mean_returns, cov_matrix):
    _, std = portfolio_annualized_performance(weights, mean_returns, cov_matrix)
    return std

def portfolio_return(weights, mean_returns, cov_matrix):
    ret, _ = portfolio_annualized_performance(weights, mean_returns, cov_matrix)
    return ret

# Cardinality constraint (number of stocks in portfolio)
def cardinality_constraint(weights, min_stocks=5, max_stocks=10):
    # Count number of stocks with weight > 0.001 (effectively non-zero)
    non_zero_count = np.sum(weights > 0.001)
    return min_stocks <= non_zero_count <= max_stocks

# Optimization function with constraints
def optimize_portfolio(objective_function, mean_returns, cov_matrix, returns_data, 
                      risk_free_rate=0.02, max_stock_weight=0.3, min_stocks=5, max_stocks=10):
    num_assets = len(mean_returns)
    
    # Constraints
    constraints = [
        {'type': 'eq', 'fun': lambda x: np.sum(x) - 1},  # Sum of weights = 1
    ]
    
    # Bounds for each stock (0 to max_stock_weight)
    bounds = tuple((0, max_stock_weight) for _ in range(num_assets))
    
    # Initial guess (equal weights)
    initial_weights = num_assets * [1./num_assets]
    
    # Optimize
    if objective_function.__name__ in ['negative_sharpe_ratio', 'negative_sortino_ratio']:
        # For Sharpe and Sortino, we minimize the negative
        if objective_function.__name__ == 'negative_sharpe_ratio':
            result = sco.minimize(objective_function, initial_weights, 
                                 args=(mean_returns, cov_matrix, risk_free_rate),
                                 method='SLSQP', bounds=bounds, constraints=constraints)
        else:
            result = sco.minimize(objective_function, initial_weights, 
                                 args=(returns_data, risk_free_rate),
                                 method='SLSQP', bounds=bounds, constraints=constraints)
    else:
        # For volatility minimization or return maximization
        result = sco.minimize(objective_function, initial_weights, 
                             args=(mean_returns, cov_matrix),
                             method='SLSQP', bounds=bounds, constraints=constraints)
    
    # Check if we need to enforce cardinality constraint
    optimized_weights = result.x
    non_zero_count = np.sum(optimized_weights > 0.001)
    
    if not (min_stocks <= non_zero_count <= max_stocks):
        # If cardinality constraint is violated, we need to use a different approach
        # This is a simplified approach - in practice, you might need a more sophisticated method
        # like genetic algorithms or mixed-integer programming
        
        # Sort weights in descending order
        sorted_indices = np.argsort(optimized_weights)[::-1]
        
        # Keep only the top max_stocks stocks
        new_weights = np.zeros(num_assets)
        for i in range(max_stocks):
            new_weights[sorted_indices[i]] = optimized_weights[sorted_indices[i]]
        
        # Normalize weights
        new_weights = new_weights / np.sum(new_weights)
        
        # Apply individual stock weight constraint
        new_weights = np.minimum(new_weights, max_stock_weight)
        new_weights = new_weights / np.sum(new_weights)
        
        optimized_weights = new_weights
    
    return optimized_weights

# Main optimization function
def portfolio_optimization(strategy, returns_data, cov_matrix, risk_free_rate=0.02):
    mean_returns = returns_data.mean()
    
    if strategy == "min_volatility":
        weights = optimize_portfolio(portfolio_volatility, mean_returns, cov_matrix, returns_data, risk_free_rate)
        performance = portfolio_volatility(weights, mean_returns, cov_matrix)
        print("Minimum Volatility Portfolio:")
        
    elif strategy == "max_return":
        weights = optimize_portfolio(portfolio_return, mean_returns, cov_matrix, returns_data, risk_free_rate)
        performance = portfolio_return(weights, mean_returns, cov_matrix)
        print("Maximum Return Portfolio:")
        
    elif strategy == "max_sharpe":
        weights = optimize_portfolio(negative_sharpe_ratio, mean_returns, cov_matrix, returns_data, risk_free_rate)
        performance = portfolio_sharpe_ratio(weights, mean_returns, cov_matrix, risk_free_rate)
        print("Maximum Sharpe Ratio Portfolio:")
        
    elif strategy == "max_sortino":
        weights = optimize_portfolio(negative_sortino_ratio, mean_returns, cov_matrix, returns_data, risk_free_rate)
        performance = portfolio_sortino_ratio(weights, returns_data.values, risk_free_rate)
        print("Maximum Sortino Ratio Portfolio:")
    
    else:
        raise ValueError("Invalid strategy. Choose from: min_volatility, max_return, max_sharpe, max_sortino")
    
    return weights, performance


# Choose your optimization objective
strategy = "max_return"  # Options: "min_volatility", "max_return", "max_sharpe", "max_sortino"

# Run optimization
optimized_weights, performance = portfolio_optimization(strategy, returns, cov_matrix, risk_free_rate)

# Display results
print(f"Performance: {performance:.4f}")
print("\nOptimized Portfolio Weights:")
for stock, weight in zip(stock_list, optimized_weights):
    if weight > 0.001:  # Only show stocks with weight > 0.1%
        print(f"{stock}: {weight:.4f} ({weight*100:.2f}%)")

# Calculate portfolio statistics
annual_return, annual_volatility = portfolio_annualized_performance(optimized_weights, mean_returns, cov_matrix)
sharpe = portfolio_sharpe_ratio(optimized_weights, mean_returns, cov_matrix, risk_free_rate)

print(f"\nAnnualized Return: {annual_return:.4f}")
print(f"Annualized Volatility: {annual_volatility:.4f}")
print(f"Sharpe Ratio: {sharpe:.4f}")